In [5]:
import ee



#####################################
#    SET ALL PARAMETERS #############
#####################################

PROJECT_DIR = "projects/ee-tymc5571-cbi-module/assets/"
GEE_PROJECT = "ee-tymc5571-cbi-module"
CRS = "EPSG:4326" #this is the CRS that the data will be exported in as an asset
EXPORT_CRS = "EPSG:5070" #this is the CRS that the data will be exported in as a geotiff
DRIVE_FOLDER = "GEE_Exports_cbi_module_forest_masks" #this is the folder in your google drive that the data will be exported to as a geotiff

#ee.Authenticate()
ee.Authenticate(
    auth_mode='notebook',  # or 'gcloud' or 'appdefault' in other contexts
    scopes=['https://www.googleapis.com/auth/earthengine',
            'https://www.googleapis.com/auth/devstorage.read_write',
            'https://www.googleapis.com/auth/drive']
    #, force = True
    #, quiet=False  # force re-authentication even if cached tokens exist
)
ee.Initialize(project=GEE_PROJECT)

# Set output region as desired
states = ee.FeatureCollection("TIGER/2018/States")

western_states = [
    'WA', 'OR', 'CA', 'ID', 'NV', 'MT', 'WY', 'UT',
    'CO', 'AZ', 'NM'#, 'ND', 'SD', 'NE', 'KS', 'OK', 'TX'
]
outputRegion = states.filter(
    ee.Filter.inList('STUSPS', western_states)
)

START_YEAR = 1984
END_YEAR = 2025
CRS = "EPSG:4326" #this is the CRS that the data will be exported in

# Data
lcms = ee.ImageCollection("USFS/GTAC/LCMS/v2024-10") 
lcpri = ee.ImageCollection("projects/sat-io/open-datasets/LCMAP/LCPRI")

SERVICE_FILE = "C:/Users/tymc5571/dev/cbi-module/config/secrets/tymc5571-utils-project-692f27c034dd.json"  # Path to service account file
DIR_DERIVED = "C:/Users/tymc5571/dev/cbi-module/data/derived/"


In [2]:
# A few functions
def remove_to_bands_append2(image, og_ic):
    """
    Removes image ID prefixes (from toBands()) using IDs from the original ImageCollection.
    
    Args:
        image (ee.Image): The image with compound band names after toBands().
        og_ic (ee.ImageCollection): The original ImageCollection before stacking.
    
    Returns:
        ee.Image: Image with cleaned band names.
    """
    ids = ee.List(og_ic.aggregate_array('system:index'))

    def clean_name(name):
        name = ee.String(name)
        def iterate_fn(id_, acc):
            acc_str = ee.String(acc)
            id_str = ee.String(id_).cat('_')
            match = acc_str.slice(0, id_str.length()).equals(id_str)
            return ee.Algorithms.If(match, acc_str.slice(id_str.length()), acc_str)
        return ids.iterate(iterate_fn, name)

    old_names = image.bandNames()
    new_names = old_names.map(lambda b: clean_name(b))
    return image.rename(new_names)

def toBands_with_projection(collection):
    collection = ee.ImageCollection(collection)
    image = collection.toBands()
    reference_img = ee.Image(collection.first())
    return image.setDefaultProjection(reference_img.projection())

In [3]:
# /////////////////////////////////////////////////////////////////////////////////
# /////////////////////////////////////////////////////////////////////////////////
# ///////////       GENERATE ANNUAL CONSERVATIVE/RELAXED FOREST MASK      /////////////////
# /////////////////////////////////////////////////////////////////////////////////
# /////////////////////////////////////////////////////////////////////////////////

# Print image collections to view if necessary
# Uncomment the following lines to print the collections
# print('LCPRI:', lcpri.getInfo())
# print('LCMS:', lcms.getInfo())

# Reclassify an image to binary (1 = forest)
def reclassify_image_binary(value):
    def inner(image):
        binary = image.eq(value).selfMask()
        return binary.copyProperties(image, ['system:time_start'])
    return inner

# AND operation between two image collections
def combine_collections_with_and(collection1, collection2, and_name):
    list1 = collection1.toList(collection1.size())
    list2 = collection2.toList(collection2.size())
    zipped = list1.zip(list2)

    def and_func(pair):
        image1 = ee.Image(ee.List(pair).get(0))
        image2 = ee.Image(ee.List(pair).get(1))
        result = image1.And(image2).rename(and_name)
        return result.copyProperties(image1, ['system:time_start'])

    return ee.ImageCollection(zipped.map(and_func))

# OR operation between two image collections
def combine_collections_with_or(collection1, collection2, or_name):
    list1 = collection1.toList(collection1.size())
    list2 = collection2.toList(collection2.size())
    zipped = list1.zip(list2)

    def or_func(pair):
        image1 = ee.Image(ee.List(pair).get(0))
        image2 = ee.Image(ee.List(pair).get(1))
        result = image1.Or(image2).rename(or_name)
        return result.copyProperties(image1, ['system:time_start'])

    return ee.ImageCollection(zipped.map(or_func))

# Add year property from system:time_start
def add_year_property(image):
    year = image.date().get('year')
    return image.set('year', year)



######################################
# OPERATE



# Filter LCPRI and LCMS collections by date
lcpri_cover = lcpri.filterDate(str(START_YEAR), str(END_YEAR))
lcms_cover = lcms.filter(ee.Filter.eq('study_area', 'CONUS')) \
                 .filterDate(str(START_YEAR), str(END_YEAR)) \
                 .select('Land_Cover')

# Reclassify forest (LCPRI=4, LCMS=1)
lcpri_for = lcpri_cover.map(reclassify_image_binary(4))
lcms_for = lcms_cover.map(reclassify_image_binary(1))

# Combine LCPRI and LCMS forests
conservative_forest = combine_collections_with_and(lcpri_for, lcms_for, 'conservativeForest') \
                        .map(add_year_property)

print('Conservative forest:', conservative_forest.aggregate_array('year').getInfo())

relaxed_forest = combine_collections_with_or(lcpri_for, lcms_for, 'relaxedForest') \
                        .map(add_year_property)

print('Relaxed forest:', relaxed_forest.aggregate_array('year').getInfo())

# Required forest year
required_forest_years = range(START_YEAR, END_YEAR)
actual_forest_years = conservative_forest.aggregate_array('year').distinct().sort().getInfo()

# Find missing forest years
missing_forest_years = set(required_forest_years) - (set(actual_forest_years))
print('Missing conservative forest years:', missing_forest_years)

# Determine available years in LCPRI and LCMS
lcpri_years = lcpri_cover.aggregate_array('system:time_start') \
    .map(lambda d: ee.Date(d).get('year')).distinct().getInfo()
lcpri_years = set(lcpri_years)
lcms_years = lcms_cover.aggregate_array('system:time_start') \
    .map(lambda d: ee.Date(d).get('year')).distinct().getInfo()
lcms_years = set(lcms_years)

available_from_lcpri = missing_forest_years.intersection(lcpri_years)
available_from_lcms = missing_forest_years.intersection(lcms_years)
available_from_either = available_from_lcpri.union(available_from_lcms)

if len(available_from_either) == 0:
    print("No missing forest years available from LCMS or LCPRI.") 
else:
    print("Missing forest years available from LCMS:", available_from_lcms)
    print("Missing forest years available from LCPRI:", available_from_lcpri)

def get_year_image(col, year):
    """Return the (binary) forest image for a given year from an ImageCollection."""
    return ee.Image(
        col.filter(ee.Filter.calendarRange(year, year, 'year')).first()
    )

final_conservative_forest = conservative_forest
final_relaxed_forest = relaxed_forest
missing_forest_years_sorted = sorted(missing_forest_years)

for y in missing_forest_years_sorted:
    src_img = None
    if y in lcpri_years:
        src_img = get_year_image(lcpri_for, y)
    elif y in lcms_years:
        src_img = get_year_image(lcms_for, y)
    
    if src_img is None:
        continue

    backfill_img_c = (
        src_img
        .rename('conservativeForest')
        .set('year', y)
    )

    backfill_img_r = (
        src_img
        .rename('relaxedForest')
        .set('year', y)
    )

    final_conservative_forest = final_conservative_forest.merge(
        ee.ImageCollection([backfill_img_c])
    )

    final_relaxed_forest = final_relaxed_forest.merge(
        ee.ImageCollection([backfill_img_r])
    )

final_years = (
    final_conservative_forest
    .aggregate_array('year')
    .distinct()
    .sort()
    .getInfo()
)
print("Final forest years:", final_years)

def rename_forest_band(img):
    year = ee.Number(img.get('year')).format('%d')
    band_name = ee.String('forest_').cat(year)
    
    img = img.rename(band_name)
    img = img.set('system:index', band_name)
    return img

final_conservative_forest_renamed = final_conservative_forest.map(rename_forest_band)
final_relaxed_forest_renamed = final_relaxed_forest.map(rename_forest_band)

conservative_forest_by_year_raw = toBands_with_projection(final_conservative_forest_renamed)
relaxed_forest_by_year_raw = toBands_with_projection(final_relaxed_forest_renamed)

conservative_forest_by_year_img = remove_to_bands_append2(
    conservative_forest_by_year_raw,
    og_ic=final_conservative_forest_renamed
)

relaxed_forest_by_year_img = remove_to_bands_append2(
    relaxed_forest_by_year_raw,
    og_ic=final_relaxed_forest_renamed
)

print("Conservative forest bands:", conservative_forest_by_year_img.bandNames().getInfo())
print("Relaxed forest bands:", relaxed_forest_by_year_img.bandNames().getInfo())

print(min(final_years))




cassetID = f"{PROJECT_DIR}conservative_forest_{str(min(final_years))}_{str(max(final_years))}"
print("Checking asset ID:", cassetID)

def asset_exists(asset_id):
    try:
        ee.data.getAsset(asset_id)
        return True
    except ee.EEException:
        return False

if asset_exists(cassetID):
    print(f"Asset already exists, skipping export: {cassetID}")
else:
    print("Exporting to asset ID:", cassetID)

    task = ee.batch.Export.image.toAsset(
        image=conservative_forest_by_year_img,
        description='Conservative forest',
        assetId=cassetID,
        scale=30,
        crs=CRS,
        region=outputRegion.geometry(),
        maxPixels=1e13
    )
    task.start()

rassetID = f"{PROJECT_DIR}relaxed_forest_{str(min(final_years))}_{str(max(final_years))}"
print("Checking asset ID:", rassetID)

if asset_exists(rassetID):
    print(f"Asset already exists, skipping export: {rassetID}")
else:
    print("Exporting to asset ID:", rassetID)

    task = ee.batch.Export.image.toAsset(
        image=relaxed_forest_by_year_img,
        description='Relaxed forest',
        assetId=rassetID,
        scale=30,
        crs=CRS,
        region=outputRegion.geometry(),
        maxPixels=1e13
    )
    task.start()

Conservative forest: [1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
Relaxed forest: [1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
Missing conservative forest years: {1984, 2024, 2022, 2023}
Missing forest years available from LCMS: {2024, 2022, 2023}
Missing forest years available from LCPRI: set()
Final forest years: [1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Conservative forest bands: ['forest_1985', 'forest_1986', 'forest_1987', 'forest_1988'

In [ ]:
# Export to drive

conservative_forest_by_year_img = ee.Image(cassetID)
relaxed_forest_by_year_img = ee.Image(rassetID)


task_drive_conservative = ee.batch.Export.image.toDrive(
    image=conservative_forest_by_year_img,
    description="conservative_forest_toDrive",
    folder=DRIVE_FOLDER,
    fileNamePrefix="forest_conservative_annual",
    region=outputRegion.geometry(),
    scale=30,
    crs=EXPORT_CRS,
    maxPixels=1e13
)
task_drive_conservative.start()


task_drive_relaxed = ee.batch.Export.image.toDrive(
    image=relaxed_forest_by_year_img,
    description="relaxed_forest_toDrive",
    folder=DRIVE_FOLDER,
    fileNamePrefix="forest_relaxed_annual",
    region=outputRegion.geometry(),
    scale=30,
    crs=EXPORT_CRS,
    maxPixels=1e13
)
task_drive_relaxed.start()

: 

In [18]:
# Support functions for export

import os
import shutil
import tempfile
import contextlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, List
import time, random

from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload

# GDAL bindings
from osgeo import gdal

gdal.UseExceptions() #get GDAL tracebacks


def _gdal_error_details(prefix: str) -> RuntimeError:
    """
    Build a RuntimeError that includes GDAL's last error message.
    """
    msg = gdal.GetLastErrorMsg()
    if msg:
        return 
        (f"{prefix}\nGDAL error:\n{msg}")
    return RuntimeError(prefix)


def list_all_files_in_folder(
    service,
    folder_id: str,
    description: Optional[str] = None,
    *,
    include_all_drives: bool = True,
    drive_id: Optional[str] = None,
    page_size: int = 1000,
) -> List[dict]:
    """
    List ALL files in a Drive folder, with pagination.
    Filters by description substring and .tif/.TIF extension.
    Returns a list of dicts with at least {'id','name'}.
    """
    # Query: in folder, not trashed, optional name contains 'description',
    # and extension contains '.tif' or '.TIF'
    name_filter = []
    if description:
        name_filter.append(f"(name contains '{description}')")
    ext_filter = "((name contains '.tif') or (name contains '.TIF'))"
    q_parts = [f"'{folder_id}' in parents", "trashed=false", ext_filter]
    if name_filter:
        q_parts.append(" and ".join(name_filter))
    q = " and ".join(q_parts)

    kwargs = dict(
        q=q,
        pageSize=page_size,
        fields="nextPageToken, files(id, name)",
        orderBy="name",
    )
    if include_all_drives:
        kwargs.update(dict(supportsAllDrives=True, includeItemsFromAllDrives=True))
        if drive_id:
            kwargs.update(dict(corpora="drive", driveId=drive_id))
        else:
            kwargs.update(dict(corpora="allDrives"))
    else:
        kwargs.update(dict(corpora="user"))

    files, page_token = [], None
    while True:
        resp = service.files().list(pageToken=page_token, **kwargs).execute()
        files.extend(resp.get("files", []))
        page_token = resp.get("nextPageToken")
        if not page_token:
            break
    return files

def _download_file_from_drive(file_id, file_name, temp_dir, service_account_file):
    try:
        # Authenticate service account
        SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
        credentials = service_account.Credentials.from_service_account_file(
            os.path.abspath(service_account_file), scopes=SCOPES
        )
        service = build('drive', 'v3', credentials=credentials)

        request = service.files().get_media(fileId=file_id)
        local_path = os.path.join(temp_dir, file_name)
        with open(local_path, 'wb') as f:
            downloader = MediaIoBaseDownload(f, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
        time.sleep(random.uniform(0.5, 1.5))  # Random delay
        print(f"Downloaded {file_name}")
        return local_path
    except Exception as e:
        print(f"Error downloading {file_name}: {e}")
        return None


def download_merge_from_drive_tif(
    description: str,
    local_filename: str,
    drive_folder: str,
    service_account_file: str,
    *,
    # Output & mosaic controls
    compress: str = "DEFLATE",          # DEFLATE|LZW|ZSTD|JPEG
    predictor: Optional[int] = None,     # 2 for integers, 3 for floats; None = let GDAL decide
    blocksize: int = 512,                # tile size for output GeoTIFF
    bigtiff: str = "YES",                # YES|IF_SAFER
    dst_srs: Optional[str] = None,       # e.g., "EPSG:5070" (if reprojection needed)
    xres: Optional[float] = None,        # set both xres & yres to enforce pixel size (e.g., 30)
    yres: Optional[float] = None,
    dst_nodata: Optional[float] = None,  # set if you need a specific nodata in output
    # Drive / listing controls
    drive_id: Optional[str] = None,      # if folder is in a Shared Drive, you can pass the Drive ID
    include_all_drives: bool = True,     # search across Shared Drives
    # Behavior controls
    check_existing: bool = True,
    n_workers: int = 4,
    verbose: bool = True,
) -> str:
    """
    Download all Drive .tif/.TIF files whose names contain `description` from `drive_folder`,
    then build a VRT and stream a mosaicked GeoTIFF to `local_filename` (no in-memory merge).
    """

    # --- Fast exit if we already have the target ---
    local_filename = os.path.abspath(local_filename)
    if check_existing and os.path.exists(local_filename):
        if verbose:
            print(f"File already exists: {local_filename}")
        return local_filename

    # --- Temp workspace ---
    temp_dir = tempfile.mkdtemp()
    vrt_path = os.path.join(temp_dir, "mosaic.vrt")
    downloaded_paths: List[str] = []

    # --- Build Drive client ---
    SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
    cred_path = os.path.abspath(service_account_file)
    credentials = service_account.Credentials.from_service_account_file(cred_path, scopes=SCOPES)
    service = build("drive", "v3", credentials=credentials)

    # Convenience flags for Shared Drives
    list_kwargs = dict(
        fields="nextPageToken, files(id, name, parents)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ) if include_all_drives else dict(fields="nextPageToken, files(id, name, parents)")

    try:
        # --- Locate the folder (paginate just in case) ---
        folder_q = (
            f"name='{drive_folder}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        )
        folder_kwargs = dict(
            q=folder_q,
            pageSize=1000,
            corpora=("drive" if drive_id else "allDrives") if include_all_drives else "user",
            driveId=drive_id,
            **list_kwargs
        )

        folders, page_token = [], None
        while True:
            resp = service.files().list(pageToken=page_token, **folder_kwargs).execute()
            folders.extend(resp.get("files", []))
            page_token = resp.get("nextPageToken")
            if not page_token:
                break

        if not folders:
            raise FileNotFoundError(
                f"Folder '{drive_folder}' not found. "
                "Ensure it exists and is shared with the service account "
                f"({credentials.service_account_email}). "
                "If it is on a Shared Drive, also ensure the service account has at least Viewer."
            )
        folder_id = folders[0]["id"]

        # --- List ALL matching .tif files in that folder (paginated) ---
        files = list_all_files_in_folder(
            service,
            folder_id,
            description=description,
            include_all_drives=include_all_drives,
            drive_id=drive_id,
            page_size=1000
        )

        if not files:
            raise FileNotFoundError(
                f"No matching .tif/.TIF files found for '{description}' in '{drive_folder}'. "
                "Check filenames and sharing."
            )

        if verbose:
            print(f"Found {len(files)} file(s). Starting download to {temp_dir} ...")

        # --- Download the files (parallel if requested) ---
        if n_workers <= 1:
            for f in files:
                path = _download_file_from_drive(f["id"], f["name"], temp_dir, service_account_file)
                if path:
                    downloaded_paths.append(path)
        else:
            with ThreadPoolExecutor(max_workers=n_workers) as ex:
                futures = [
                    ex.submit(_download_file_from_drive, f["id"], f["name"], temp_dir, service_account_file)
                    for f in files
                ]
                for fut in as_completed(futures):
                    p = fut.result()
                    if p:
                        downloaded_paths.append(p)

        if not downloaded_paths:
            raise RuntimeError("Downloads finished but no files were saved.")

        if verbose:
            print(f"Building VRT from {len(downloaded_paths)} tile(s) ...")

        # --- Build VRT (lightweight, no data copy) ---
        vrt_options = {}
        if dst_nodata is not None:
            vrt_options.update(dict(srcNodata=dst_nodata, VRTNodata=dst_nodata))

        vrt = gdal.BuildVRT(vrt_path, downloaded_paths, **vrt_options)
        if vrt is None:
            raise _gdal_error_details("GDAL BuildVRT failed.")
        vrt = None  # flush

        # --- Stream out to GeoTIFF (no giant array in RAM) ---
        co = [
            f"BIGTIFF={bigtiff}",
            "TILED=YES",
            f"COMPRESS={compress.upper()}",
            f"BLOCKXSIZE={int(blocksize)}",
            f"BLOCKYSIZE={int(blocksize)}",
            "OVERVIEWS=AUTO", #pyramids
            "RESAMPLING=AVERAGE",
            "NUM_THREADS=ALL_CPUS"
        ]
        if predictor is not None:
            co.append(f"PREDICTOR={int(predictor)}")

        needs_warp = any(v is not None for v in (dst_srs, xres, yres, dst_nodata))
        if needs_warp:
            if verbose:
                print("Warping (streamed) to target grid...")
            warp_kwargs = dict(
                format="GTiff",
                creationOptions=co,
                multithread=True,
            )
            if dst_srs:
                warp_kwargs["dstSRS"] = dst_srs
            if xres and yres:
                warp_kwargs["xRes"] = float(xres)
                warp_kwargs["yRes"] = float(yres)
                warp_kwargs["targetAlignedPixels"] = True
            if dst_nodata is not None:
                warp_kwargs["dstNodata"] = dst_nodata

            out_ds = gdal.Warp(local_filename, vrt_path, **warp_kwargs)
            if out_ds is None:
                raise _gdal_error_details("GDAL Warp failed.")
            out_ds = None
        else:
            if verbose:
                print("Translating VRT to tiled, compressed GeoTIFF (streamed)...")
            out_ds = gdal.Translate(local_filename, vrt_path, creationOptions=co)
            if out_ds is None:
                raise _gdal_error_details("GDAL Translate failed.")
            out_ds = None

        if verbose:
            print(f"Final mosaicked GeoTIFF saved to: {local_filename}")
        return local_filename

    except HttpError as e:
        msg = str(e)
        if "accessNotConfigured" in msg or "has not been used in project" in msg:
            raise RuntimeError(
                "Google Drive API is disabled for this service account’s project.\n"
                "Enable the **Google Drive API** in GCP → APIs & Services → Library, then retry."
            ) from e
        raise
    except KeyboardInterrupt:
        print("Interrupted by user. Cleaning up and exiting.")
        raise
    finally:
        # Best-effort cleanup
        try:
            shutil.rmtree(temp_dir)
            if verbose:
                print(f"Cleaned up temporary directory: {temp_dir}")
        except Exception as cleanup_err:
            print(f"Error during cleanup: {cleanup_err}")


def download_merge_from_drive_cog(
    description: str,
    local_filename: str,
    drive_folder: str,
    service_account_file: str,
    *,
    # Output & mosaic controls
    compress: str = "DEFLATE",          # DEFLATE|LZW|ZSTD|JPEG
    predictor: Optional[int] = None,     # 2 for integers, 3 for floats; None = let GDAL decide
    blocksize: int = 512,                # tile size for output COG (both x and y)
    bigtiff: str = "YES",                # YES|IF_SAFER
    dst_srs: Optional[str] = None,       # e.g., "EPSG:5070" (if reprojection needed)
    xres: Optional[float] = None,        # set both xres & yres to enforce pixel size (e.g., 30)
    yres: Optional[float] = None,
    dst_nodata: Optional[float] = None,  # set if you need a specific nodata in output
    # Drive / listing controls
    drive_id: Optional[str] = None,      # if folder is in a Shared Drive, you can pass the Drive ID
    include_all_drives: bool = True,     # search across Shared Drives
    # Behavior controls
    check_existing: bool = True,
    n_workers: int = 4,
    verbose: bool = True,
) -> str:
    """
    Download all Drive .tif/.TIF files whose names contain `description` from `drive_folder`,
    then build a VRT and stream a mosaicked GeoTIFF to `local_filename` (no in-memory merge).
    """

    # --- Fast exit if we already have the target ---
    local_filename = os.path.abspath(local_filename)
    if check_existing and os.path.exists(local_filename):
        if verbose:
            print(f"File already exists: {local_filename}")
        return local_filename

    # --- Temp workspace ---
    temp_dir = tempfile.mkdtemp()
    vrt_path = os.path.join(temp_dir, "mosaic.vrt")
    downloaded_paths: List[str] = []

    # --- Build Drive client ---
    SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
    cred_path = os.path.abspath(service_account_file)
    credentials = service_account.Credentials.from_service_account_file(cred_path, scopes=SCOPES)
    service = build("drive", "v3", credentials=credentials)

    # Convenience flags for Shared Drives
    list_kwargs = dict(
        fields="nextPageToken, files(id, name, parents)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ) if include_all_drives else dict(fields="nextPageToken, files(id, name, parents)")

    try:
        # --- Locate the folder (paginate just in case) ---
        folder_q = (
            f"name='{drive_folder}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        )
        folder_kwargs = dict(
            q=folder_q,
            pageSize=1000,
            corpora=("drive" if drive_id else "allDrives") if include_all_drives else "user",
            driveId=drive_id,
            **list_kwargs
        )

        folders, page_token = [], None
        while True:
            resp = service.files().list(pageToken=page_token, **folder_kwargs).execute()
            folders.extend(resp.get("files", []))
            page_token = resp.get("nextPageToken")
            if not page_token:
                break

        if not folders:
            raise FileNotFoundError(
                f"Folder '{drive_folder}' not found. "
                "Ensure it exists and is shared with the service account "
                f"({credentials.service_account_email}). "
                "If it is on a Shared Drive, also ensure the service account has at least Viewer."
            )
        folder_id = folders[0]["id"]

        # --- List ALL matching .tif files in that folder (paginated) ---
        files = list_all_files_in_folder(
            service,
            folder_id,
            description=description,
            include_all_drives=include_all_drives,
            drive_id=drive_id,
            page_size=1000
        )

        if not files:
            raise FileNotFoundError(
                f"No matching .tif/.TIF files found for '{description}' in '{drive_folder}'. "
                "Check filenames and sharing."
            )

        if verbose:
            print(f"Found {len(files)} file(s). Starting download to {temp_dir} ...")

        # --- Download the files (parallel if requested) ---
        if n_workers <= 1:
            for f in files:
                path = _download_file_from_drive(f["id"], f["name"], temp_dir, service_account_file)
                if path:
                    downloaded_paths.append(path)
        else:
            with ThreadPoolExecutor(max_workers=n_workers) as ex:
                futures = [
                    ex.submit(_download_file_from_drive, f["id"], f["name"], temp_dir, service_account_file)
                    for f in files
                ]
                for fut in as_completed(futures):
                    p = fut.result()
                    if p:
                        downloaded_paths.append(p)

        if not downloaded_paths:
            raise RuntimeError("Downloads finished but no files were saved.")

        if verbose:
            print(f"Building VRT from {len(downloaded_paths)} tile(s) ...")

        # --- Build VRT (lightweight, no data copy) ---
        vrt_options = {}
        if dst_nodata is not None:
            vrt_options.update(dict(srcNodata=dst_nodata, VRTNodata=dst_nodata))

        vrt = gdal.BuildVRT(vrt_path, downloaded_paths, **vrt_options)
        if vrt is None:
            raise _gdal_error_details("GDAL BuildVRT failed.")
        vrt = None  # flush

        # --- Stream out to Cloud-Optimized GeoTIFF (COG) ---
        # COG creation options (driver: "COG")
        co_cog = [
            f"COMPRESS={compress.upper()}",     # e.g., DEFLATE|LZW|ZSTD|JPEG
            f"BLOCKSIZE={int(blocksize)}",      # one value for both X/Y, typical 512
            f"NUM_THREADS={'ALL_CPUS'}",
            f"BIGTIFF={bigtiff}",               # YES|IF_SAFER
            "OVERVIEW_RESAMPLING=AVERAGE",      # build internal overviews with this kernel
            # Optional: "SPARSE_OK=YES",       # can help on sparse rasters
        ]
        if predictor is not None:
            co_cog.append(f"PREDICTOR={int(predictor)}")  # 2=int, 3=float (for DEFLATE/LZW)

        needs_warp = any(v is not None for v in (dst_srs, xres, yres, dst_nodata))

        if needs_warp:
            if verbose:
                print("Warping to target grid (VRT) and writing COG...")
            # Warp to a VRT (no data copy), then translate that VRT → COG
            warped_vrt = os.path.join(temp_dir, "warped.vrt")
            warp_kwargs = dict(
                format="VRT",           # keep it as VRT to avoid creating a big intermediate
                multithread=True,
            )
            if dst_srs:
                warp_kwargs["dstSRS"] = dst_srs
            if xres and yres:
                warp_kwargs["xRes"] = float(xres)
                warp_kwargs["yRes"] = float(yres)
                warp_kwargs["targetAlignedPixels"] = True
            if dst_nodata is not None:
                warp_kwargs["dstNodata"] = dst_nodata

            out_warp = gdal.Warp(warped_vrt, vrt_path, **warp_kwargs)
            if out_warp is None:
                raise _gdal_error_details("GDAL Warp (to VRT) failed.")
            out_warp = None

            # Final: VRT → COG
            out_ds = gdal.Translate(
                local_filename, warped_vrt,
                format="COG",
                creationOptions=co_cog
            )
            if out_ds is None:
                raise _gdal_error_details("GDAL Translate to COG failed.")
            out_ds = None

        else:
            if verbose:
                print("Translating VRT directly to COG (internal overviews)...")
            out_ds = gdal.Translate(
                local_filename, vrt_path,
                format="COG",
                creationOptions=co_cog
            )
            if out_ds is None:
                raise _gdal_error_details("GDAL Translate to COG failed.")
            out_ds = None

        if verbose:
            print(f"Final COG saved to: {local_filename}")
        return local_filename

    except HttpError as e:
        msg = str(e)
        if "accessNotConfigured" in msg or "has not been used in project" in msg:
            raise RuntimeError(
                "Google Drive API is disabled for this service account’s project.\n"
                "Enable the **Google Drive API** in GCP → APIs & Services → Library, then retry."
            ) from e
        raise
    except KeyboardInterrupt:
        print("Interrupted by user. Cleaning up and exiting.")
        raise
    finally:
        # Best-effort cleanup
        try:
            shutil.rmtree(temp_dir)
            if verbose:
                print(f"Cleaned up temporary directory: {temp_dir}")
        except Exception as cleanup_err:
            print(f"Error during cleanup: {cleanup_err}")

In [20]:
c_path = os.path.join(DIR_DERIVED, "conservative_forest_annual.tif")

if not os.path.exists(c_path):
    download_merge_from_drive_cog(
        description="forest_conservative_annual",
        local_filename=c_path,
        drive_folder=DRIVE_FOLDER,
        service_account_file=SERVICE_FILE,
        compress="ZSTD",         
        predictor=2,                # 3 = best for float32 data (like indices or continuous values); 2 = best for integer
        blocksize=512,              # tile size; 256–512 is good default
        bigtiff="YES",              # ensures large mosaics don’t overflow 4 GB
        #dst_srs="EPSG:5070",        # optional: force reprojection to your CONUS Albers grid
        #xres=30, yres=30,           # optional: force 30 m pixels
        #dst_nodata=-9999,           # optional: set output NoData if needed
        check_existing=True,
        n_workers=8,                # parallel downloads from Drive
        include_all_drives=True,    # safer if GEE_Exports is on a Shared Drive
        verbose=True
    )

r_path = os.path.join(DIR_DERIVED, "relaxed_forest_annual.tif")

if not os.path.exists(r_path):
    download_merge_from_drive_cog(
        description="forest_relaxed_annual",
        local_filename=r_path,
        drive_folder=DRIVE_FOLDER,
        service_account_file=SERVICE_FILE,
        compress="ZSTD",         
        predictor=2,                # 3 = best for float32 data (like indices or continuous values); 2 = best for integer
        blocksize=512,              # tile size; 256–512 is good default
        bigtiff="YES",              # ensures large mosaics don’t overflow 4 GB
        #dst_srs="EPSG:5070",        # optional: force reprojection to your CONUS Albers grid
        #xres=30, yres=30,           # optional: force 30 m pixels
        #dst_nodata=-9999,           # optional: set output NoData if needed
        check_existing=True,
        n_workers=8,                # parallel downloads from Drive
        include_all_drives=True,    # safer if GEE_Exports is on a Shared Drive
        verbose=True
    )

Found 42 file(s). Starting download to C:\Users\tymc5571\AppData\Local\Temp\tmpe9nfk450 ...
Downloaded forest_conservative_annual-0000000000-0000052480.tif
Downloaded forest_conservative_annual-0000000000-0000041984.tif
Downloaded forest_conservative_annual-0000000000-0000000000.tif
Downloaded forest_conservative_annual-0000000000-0000031488.tif
Downloaded forest_conservative_annual-0000010496-0000010496.tif
Downloaded forest_conservative_annual-0000010496-0000041984.tif
Downloaded forest_conservative_annual-0000010496-0000000000.tif
Downloaded forest_conservative_annual-0000010496-0000052480.tif
Downloaded forest_conservative_annual-0000000000-0000010496.tif
Downloaded forest_conservative_annual-0000000000-0000020992.tif
Downloaded forest_conservative_annual-0000010496-0000031488.tif
Downloaded forest_conservative_annual-0000010496-0000020992.tif
Downloaded forest_conservative_annual-0000020992-0000010496.tif
Downloaded forest_conservative_annual-0000020992-0000052480.tif
Downloaded f

In [16]:
from osgeo import gdal
print(gdal.__version__)


drivers = gdal.GetDriverByName("GTiff").GetMetadata()
print(drivers.get("DMD_CREATIONOPTIONLIST"))

print(
    gdal.GetDriverByName("COG")
        .GetMetadata()
        .get("DMD_CREATIONOPTIONLIST")
)

3.11.4
<CreationOptionList>   <Option name='COMPRESS' type='string-select'>       <Value>NONE</Value>       <Value>LZW</Value>       <Value>PACKBITS</Value>       <Value>JPEG</Value>       <Value>CCITTRLE</Value>       <Value>CCITTFAX3</Value>       <Value>CCITTFAX4</Value>       <Value>DEFLATE</Value>       <Value>LZMA</Value>       <Value>ZSTD</Value>       <Value>WEBP</Value>       <Value>LERC</Value>       <Value>LERC_DEFLATE</Value>       <Value>LERC_ZSTD</Value>   </Option>   <Option name='PREDICTOR' type='int' description='Predictor Type (1=default, 2=horizontal differencing, 3=floating point prediction)' default='1'/>   <Option name='DISCARD_LSB' type='string' description='Number of least-significant bits to set to clear as a single value or comma-separated list of values for per-band values'/>   <Option name='JPEG_QUALITY' type='int' description='JPEG quality 1-100' min='1' max='100' default='75'/>   <Option name='JPEGTABLESMODE' type='int' description='Content of JPEGTABLES t